# RIGAD

**Point this at a folder of draft papers. It tells you where to submit them and who to talk to.**

For each draft:
- which **ICIS 2026 tracks** it fits, with a confidence rating
- which **EUTOPIA researchers** at *other* institutions work on related things

With five or more drafts it also proposes **working groups**.

---

### Setting up

Put your drafts in a folder — `.pdf`, `.docx`, `.txt` or `.md`, in any mix:

```
my-drafts/                    my-drafts/
  alice.pdf                     gothenburg/alice.pdf
  bob.docx                      warwick/bob.docx
  chen.pdf                      essec/chen.pdf
```

Use subfolders if you know people's institutions and want groups to be **required to cross them**. A flat folder works too — grouping then balances topics only.

Nothing leaves your machine, and no account or API key is needed.

In [ ]:
# Make `rigad` importable however this notebook was opened.
# Jupyter often runs on a different Python than the project's environment,
# so `import rigad` can fail even when everything is installed. Adding the
# repository's src/ directory fixes that without installing anything.
import sys
from pathlib import Path

here = Path.cwd().resolve()
repo = here if (here / 'src' / 'rigad').is_dir() else next(
    (p for p in here.parents if (p / 'src' / 'rigad').is_dir()), None
)
if repo:
    sys.path.insert(0, str(repo / 'src'))

try:
    import rigad
    print(f'RIGAD {rigad.__version__} ready')
except ImportError as missing:
    # Dependencies live in the project environment. Rather than install into
    # whichever Python Jupyter happens to be using, say what to do.
    print(f'Cannot load RIGAD: {missing}\n')
    print(f'This notebook is running on: {sys.executable}\n')
    print('Switch the kernel to "RIGAD (project venv)" — Kernel > Change Kernel —')
    print('or start Jupyter from the project environment:\n')
    print('    uv run jupyter lab notebooks/RIGAD.ipynb\n')
    print('If the kernel is not listed, register it once with:\n')
    print('    uv run python -m ipykernel install --user --name rigad \\\n')
    print('        --display-name "RIGAD (project venv)"')

In [ ]:
DRAFTS_FOLDER = "ddd/"   # <-- change this to your folder

In [ ]:
from rigad import analyse

result = analyse(DRAFTS_FOLDER)
result.show()

---

## Reading the output

**Confidence** is the gap between the best-fitting track and the runner-up, banded using the distribution measured over 1,463 real papers:

| | meaning |
|---|---|
| **HIGH** | top quartile of margins — the leading track is clearly ahead |
| **MODERATE** | above the median |
| **LOW** | the top tracks are close together |

A **LOW** rating is information, not a failure. It usually means the draft genuinely straddles two communities, and the choice should be made on which audience you want rather than on which number is fractionally larger. In the evaluation, wide-margin drafts were matched more consistently (0.425) than narrow-margin ones (0.333), which is why the rating is shown at all.

**Matched on** tells you *why* a track was suggested. A named topic of interest is specific, strong evidence. "The track's overall scope" means the draft fits the track broadly without hitting any of its listed topics — weaker, and worth a second look.

**Mentors** are drawn from a pool of 76 researchers at the three partner IS departments, and always exclude the draft's own institution — you already know who works down the corridor.

## Options

Everything is optional. Defaults are chosen so the cell above just works.

In [ ]:
from rigad import analyse

result = analyse(DRAFTS_FOLDER)
result.show()

## Working with the results directly

`result.results` is a plain list, one entry per draft — useful for exporting to a spreadsheet or feeding into your own workflow.

In [ ]:
import csv

with open("rigad_recommendations.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["file", "institution", "top track", "score", "confidence", "top mentor"])
    for r in result.results:
        writer.writerow([
            r.draft.path.name,
            r.draft.institution or "",
            r.tracks[0].track.name,
            f"{r.tracks[0].score:.3f}",
            r.confidence,
            r.mentors[0].mentor.name if r.mentors else "",
        ])

print("wrote rigad_recommendations.csv")
for r in result.results[:5]:
    print(f"  {r.draft.path.name[:38]:40s} {r.confidence:9s} {r.tracks[0].track.name[:34]}")

---

## Using a different conference

Nothing here is specific to ICIS. A track file is just JSON:

```json
{"conference": "Your Conference 2027",
 "tracks": [
   {"id": "a",
    "name": "Track A",
    "description": "What belongs in this track…",
    "topics": ["Topic of interest one", "Topic of interest two"]}
 ]}
```

Save it under `data/tracks/` and pass `tracks_file=` above. The topics list matters: drafts match tracks through individual topics, so a track with well-written topics of interest gets more specific — and better-explained — matches than one with only a prose blurb.

## Limitations worth knowing

- Recommendations come from **semantic similarity**, not from any model of what reviewers actually accept.
- Evaluation used published papers, which are cleaner than in-progress drafts.
- The mentor pool covers three IS departments (Gothenburg, Warwick, ESSEC), not all of EUTOPIA.
- Scanned PDFs with no text layer are skipped — they are reported, not silently dropped.

Method and evaluation: [`docs/`](../docs). Everything is MIT licensed.